In [54]:
import pandas as pd
import os

#  check json file

In [55]:
import pandas as pd
import os

file_path = "../include/data/raw_crimes_20260310_112555.json"

if not os.path.exists(file_path):
    raise FileNotFoundError(f"Le fichier {file_path} est introuvable.")

# Charger le JSON en DataFrame
try:
    df = pd.read_json(file_path)
except ValueError as e:
    print(f"Erreur de lecture du fichier JSON : {e}")

In [56]:
df.head()

,id,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,...,location,:@computed_region_awaf_s7ux,:@computed_region_6mkv_f3dw,:@computed_region_vrxf_vc4k,:@computed_region_bdys_3d7i,:@computed_region_43wa_7qmu,:@computed_region_rpca_8um6,:@computed_region_d9mm_jgwp,:@computed_region_d3ds_rm58,:@computed_region_8hcu_yrd4
0,14125939,JK167780,2026-03-01,010XX W CATALPA AVE,0486,BATTERY,DOMESTIC BATTERY SIMPLE,APARTMENT,False,True,...,"{'latitude': '41.981847263', 'longitude': '-87...",40.0,22616.0,76.0,145.0,16.0,15.0,2.0,47.0,48.0
1,14128921,JK171487,2026-03-01,098XX S MICHIGAN AVE,0560,ASSAULT,SIMPLE,RESIDENCE,False,True,...,"{'latitude': '41.716091748', 'longitude': '-87...",31.0,21861.0,45.0,567.0,43.0,19.0,10.0,250.0,11.0
2,14125373,JK167205,2026-03-01,007XX W 59TH ST,1320,CRIMINAL DAMAGE,TO VEHICLE,STREET,False,False,...,"{'latitude': '41.787053762', 'longitude': '-87...",53.0,21559.0,66.0,113.0,4.0,11.0,17.0,135.0,16.0
3,14124313,JK165953,2026-03-01,027XX N KILBOURN AVE,1320,CRIMINAL DAMAGE,TO VEHICLE,STREET,False,False,...,"{'latitude': '41.931117066', 'longitude': '-87...",7.0,22615.0,21.0,439.0,17.0,2.0,6.0,181.0,31.0
4,14124113,JK165627,2026-03-01,088XX S BURLEY AVE,1310,CRIMINAL DAMAGE,TO PROPERTY,APARTMENT,False,False,...,"{'latitude': '41.734847232', 'longitude': '-87...",43.0,21202.0,42.0,509.0,47.0,25.0,19.0,239.0,6.0


# Data quality with soda : first check 

In [57]:
import pandas as pd
import duckdb
import os
from soda_duckdb import DuckDBDataSource
from soda_core.contracts import verify_contract_locally

# Charger le JSON en DataFrame
file_path = "../include/data/raw_crimes_20260310_112555.json"
if not os.path.exists(file_path):
    raise FileNotFoundError(f"Le fichier {file_path} est introuvable.")
df = pd.read_json(file_path)

# Connexion DuckDB
conn = duckdb.connect(database=":memory:")
cursor = conn.cursor()

# Enregistrer le DataFrame comme vue DuckDB (args positionnels obligatoires)
cursor.register("chicago_crime", df)

# Créer la datasource Soda
duckdb_ds = DuckDBDataSource.from_existing_cursor(cursor, name="duckdb")

# Chemin du fichier YAML
contract_file_path = "../soda_scan/soda_rules_firstcheck.yml"
if not os.path.exists(contract_file_path):
    raise FileNotFoundError(f"Le fichier {contract_file_path} est introuvable.")

# Vérification
result = verify_contract_locally(
    data_sources=[duckdb_ds],
    contract_file_path=contract_file_path
)


In [61]:
# Résumé lisible
summary = {}

for contract_result in result.contract_verification_results:
    for check_result in contract_result.check_results:
        col_name = getattr(check_result.check, "column_name", None) or "Dataset"
        check_name = getattr(check_result.check, "name", None)
        outcome_str = str(check_result.outcome)  # ex: "CheckOutcome.PASSED"

        if col_name not in summary:
            summary[col_name] = []

        detail = getattr(check_result, "value", None)

        summary[col_name].append({
            "check": check_name,
            "outcome": outcome_str,
            "detail": detail
        })

# Afficher le résumé lisible
for col, checks in summary.items():
    print(f"Colonne : {col}")
    for c in checks:
        status = c['outcome']
        if "FAILED" in status or "WARNING" in status:
            print(f"  - Check : {c['check']}, Statut : {status}")
            if c['detail'] is not None:
                print(f"    Détail : {c['detail']}")
        else:
            print(f"  - Check : {c['check']}, Statut : {status}")
    print("-" * 50)

# Résumé global
all_checks = [
    check_result
    for contract_result in result.contract_verification_results
    for check_result in contract_result.check_results
]

total = len(all_checks)
passed = sum(1 for c in all_checks if "PASSED" in str(c.outcome))
failed = sum(1 for c in all_checks if "FAILED" in str(c.outcome))
warning = sum(1 for c in all_checks if "WARNING" in str(c.outcome))

print(f"\nTotal checks : {total}")
print(f"Passés        : {passed}")
print(f"Échoués       : {failed}")
print(f"Avertissements: {warning}")


Colonne : id
  - Check : No missing values, Statut : CheckOutcome.PASSED
  - Check : No duplicate values, Statut : CheckOutcome.PASSED
  - Check : No invalid values, Statut : CheckOutcome.PASSED
--------------------------------------------------
Colonne : case_number
  - Check : No missing values, Statut : CheckOutcome.PASSED
--------------------------------------------------
Colonne : date
  - Check : No missing values, Statut : CheckOutcome.PASSED
--------------------------------------------------
Colonne : year
  - Check : No missing values, Statut : CheckOutcome.PASSED
  - Check : No invalid values, Statut : CheckOutcome.PASSED
--------------------------------------------------
Colonne : updated_on
  - Check : No missing values, Statut : CheckOutcome.PASSED
--------------------------------------------------
Colonne : block
  - Check : No missing values, Statut : CheckOutcome.PASSED
--------------------------------------------------
Colonne : beat
  - Check : No missing values, Stat